### Carga de librerías

In [1]:
import numpy as np
import pandas as pd
import os

### Configuración de rutas

In [3]:
BASE_DIR = '../'

PATH_TRAIN_SPLIT = os.path.join(BASE_DIR, 'work/split/train_split.csv')
PATH_TEST_SPLIT  = os.path.join(BASE_DIR, 'work/split/test_split.csv')

PATH_TRAIN_CLEAN = os.path.join(BASE_DIR, 'work/cleaned/train_clean.csv')
PATH_TEST_CLEAN  = os.path.join(BASE_DIR, 'work/cleaned/test_clean.csv')

### Carga del split

In [4]:
train = pd.read_csv(PATH_TRAIN_SPLIT)
test  = pd.read_csv(PATH_TEST_SPLIT)

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')

Train shape: (11994, 24)
Test  shape: (2999, 24)


## Data Cleaning

Limpieza aplicada **siempre sobre train primero**, luego el mismo criterio al test para evitar data leakage.

| Columna | Problema | Solución |
|---|---|---|
| Age | Outliers > 180 meses | Imputar con mediana de train |
| Name | Nulos (~5%) | Flag  + rellenar con cadena vacía |
| Description | Nulos (~1%) | Flag  + rellenar con cadena vacía |
| Fee | Outliers extremos (hasta 3000) | Capear en percentil 99 de train |

In [5]:
# Resumen de calidad de datos post-split
def quality_summary(df, name):
    summary = pd.DataFrame({
        'Tipo':        [df[c].dtype for c in df.columns],
        '% Nulos':     [(df[c].isnull().mean() * 100).round(2) for c in df.columns],
        'Cardinalidad':[df[c].nunique() for c in df.columns],
    }, index=df.columns)
    print('\n=== ' + name + ' (' + str(df.shape[0]) + ' filas) ===')
    print(summary[summary['% Nulos'] > 0].sort_values('% Nulos', ascending=False).to_string())

quality_summary(train, "TRAIN")
quality_summary(test,  "TEST")


=== TRAIN (11994 filas) ===
               Tipo  % Nulos  Cardinalidad
Name         object     8.44          7433
Description  object     0.08         11285

=== TEST (2999 filas) ===
               Tipo  % Nulos  Cardinalidad
Name         object     8.44          2231
Description  object     0.10          2889


### 1. Outliers en Age

El EDA detectó valores de Age de hasta 255 meses (~21 años), biológicamente improbables.  
Se imputan los valores > 180 meses con la **mediana calculada sobre train** únicamente para evitar data leakage.

In [7]:
# Mediana calculada solo en train (sin outliers)
age_median_train = train.loc[train["Age"] <= 180, "Age"].median()
age_median_test = test.loc[test["Age"] <= 180, "Age"].median()

train.loc[train["Age"] > 180, "Age"] = np.nan
test.loc[test["Age"]  > 180, "Age"] = np.nan

train["Age"] = train["Age"].fillna(age_median_train)
test["Age"]  = test["Age"].fillna(age_median_test)

print(f"Mediana de Age (train, <= 180 meses): {age_median_train:.0f} meses")
print(f"Mediana de Age (test, <= 180 meses): {age_median_test:.0f} meses")
print(f"Train - Age max: {train['Age'].max()}, nulos restantes: {train['Age'].isnull().sum()}")
print(f"Test  - Age max: {test['Age'].max()},  nulos restantes: {test['Age'].isnull().sum()}")

Mediana de Age (train, <= 180 meses): 3 meses
Mediana de Age (test, <= 180 meses): 3 meses
Train - Age max: 180.0, nulos restantes: 0
Test  - Age max: 144.0,  nulos restantes: 0


### 2. Columna Name (nulos)

No todos los anunciantes le ponen nombre a la mascota → nulos esperables.  
Se crea el flag binario  (1 = tiene nombre, 0 = no tiene) y se rellenan los nulos con cadena vacía.

In [10]:
train["has_name"] = train["Name"].notna().astype(int)
test["has_name"]  = test["Name"].notna().astype(int)

train["Name"] = train["Name"].fillna("")
test["Name"]  = test["Name"].fillna("")

print(f"Train - % con nombre: {train['has_name'].mean()*100:.1f}%")
print(f"Test  - % con nombre: {test['has_name'].mean()*100:.1f}%")

Train - % con nombre: 91.6%
Test  - % con nombre: 91.6%


### 3. Columna Description (nulos)

Igual que Name, no todos los anunciantes completan la descripción.  
Se crea el flag  y se rellenan los nulos con cadena vacía.

In [9]:
train["has_description"] = train["Description"].notna().astype(int)
test["has_description"]  = test["Description"].notna().astype(int)

train["Description"] = train["Description"].fillna("")
test["Description"]  = test["Description"].fillna("")

print(f"Train - % con descripción: {train['has_description'].mean()*100:.1f}%")
print(f"Test  - % con descripción: {test['has_description'].mean()*100:.1f}%")

Train - % con descripción: 99.9%
Test  - % con descripción: 99.9%


### 4. Outliers en Fee

Fee tiene valores extremos de hasta 3000 RM (adopciones de raza pura, muy pocos casos).  
Se capea en el **percentil 99 de train** para reducir la influencia de outliers sin eliminar registros.

In [8]:
fee_cap_train = train["Fee"].quantile(0.99)
fee_cap_test = test["Fee"].quantile(0.99)

train["Fee"] = train["Fee"].clip(upper=fee_cap_train)
test["Fee"]  = test["Fee"].clip(upper=fee_cap_test)

print(f"Fee capeado en Train: {fee_cap_train:.1f} RM")
print(f"Fee capeado en Test: {fee_cap_test:.1f} RM")
print(f"Train - Fee max: {train['Fee'].max():.1f}, media: {train['Fee'].mean():.2f}")
print(f"Test  - Fee max: {test['Fee'].max():.1f},  media: {test['Fee'].mean():.2f}")

Fee capeado en Train: 350.0 RM
Fee capeado en Test: 350.6 RM
Train - Fee max: 350.0, media: 19.32
Test  - Fee max: 350.6,  media: 19.09


# SACAR LOS ID

### 5. Guardar datasets limpios

In [11]:
os.makedirs(os.path.join(BASE_DIR, "work/cleaned"), exist_ok=True)

PATH_TRAIN_CLEAN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TEST_CLEAN  = os.path.join(BASE_DIR, "work/cleaned/test_clean.csv")

train.to_csv(PATH_TRAIN_CLEAN, index=False)
test.to_csv(PATH_TEST_CLEAN, index=False)

print(f"Train limpio guardado: {PATH_TRAIN_CLEAN}  shape={train.shape}")
print(f"Test limpio guardado:  {PATH_TEST_CLEAN}   shape={test.shape}")

Train limpio guardado: ../work/cleaned/train_clean.csv  shape=(11994, 26)
Test limpio guardado:  ../work/cleaned/test_clean.csv   shape=(2999, 26)
